## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [25]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field

In [26]:
load_dotenv(override=True)

True

In [27]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [28]:
instructions = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [29]:
# These URLs from the model provider provides the API endpoint that talks in the same way with OpenAI, so we can just point our code to it. 
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [30]:
# We use AsyncOpenAI because we are using asyncio to run the code
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

STEP 3: Create a model object

In [31]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client) # The strongest open source model up to date
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)

In [32]:
# When we call OpenAI LLM, if we use model=a string, it will understant that the model name is an OpenAI model. When we use a model from another provider, we have to create a model object using the Base URL and then parse that model object to the model= in the Agent object. 
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [33]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [34]:
# from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        # push(f"Subject: {subject}\n\n{text_body}")
        print("Email sent successfully (not using email sender or push service in this example)")

In [35]:
# send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [36]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [37]:
tools = [tool1, tool2, tool3, send_email_tool]

In [38]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")

In [39]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

I couldn’t generate the three drafts because all three sales writer tools failed due to authentication/API key errors.


## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
   When a LLM generates text, it does not generat the next token. It predicts the next token based on probability. In "constrained decoding", LLM zero out any next token that could break the spec and does not match the JSON schema. It sounds simple but very clever. So that very likely, almost certainly, the output will always match the expected JSON schema. 
2. Not all providers support Structured Outputs.

Best practice is to keep the JSON schema simple. You want LLM to generate simple LLM schema instead of complex nested JSOn schema. You can then use Python code to nest and structure the JSON schema afterwards. 

In [40]:
# We want to give the field description in the natural lanaguage so that LLM can understand. 
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [41]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [42]:
email = """
Hi [first_name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Ed
"""

In [43]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
result = await Runner.run(checker, email)

In [44]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [45]:
review.is_professional

False

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

If you switch to another SDK, or the OpenAI SDK updates, it's hard to debug because the guardrails are built very deeplyin the OpenAI SDK.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [46]:
# You can also use Python code comparison to implement guardrails. But LLM output is best judged by the LLM itself. So it's very common to use LLM for guardrails.
# ctx is the additional context that we pass to the LLM. 
# You need to pass in an agent because you might want different guardrails for different agents. 
# message is the actual message that we are checking. 
# This approach of implementing guardrails is within the OpenAI Agent SDK.

@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem) # If tripwire_triggered is true, the output will be an executional error. "OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire"

In [47]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model="openai/gpt-4o-mini", output_guardrails=[email_guardrail])

In [48]:
result = await Runner.run(sales_agent_cowboy, "Write a cold sales email")
result.final_output

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [49]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model="openai/gpt-4o-mini")
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)


Subject: Saddle Up for Better Customer Engagement!

Howdy there,

I reckon it’s time to lasso your customer engagement and ride into the sunset of success! Here at ComplAI, we’ve rustled up a mighty fine solution to help you connect with your customers like never before.

Imagine your team wranglin’ customer queries faster than a jackrabbit on a date. Our AI-driven platform makes it easier than a Sunday mornin’ ride to boost your response times and satisfaction rates. 

How ‘bout we set up a quick chat to see how we can help you ride the trail to better customer relationships? 

Wrangle me back if you’re interested, and we’ll set a time! 

Yours in the saddle,  
[Your Name]  
Sales Manager, ComplAI  
[Your Phone Number]  
[Your Email]  

P.S. No need to be shy; let’s corral your challenges together!


In [50]:
# Guardrails can be implemented simply in a just a runner.run() instead of a voodoo tripwire in the framework. 
# This approach is not within the OpenAI Agent SDK. You can use it with any framework. Simply just put a guardrail when calling a tool. If you use someone else's tool, then just wrap their tool within your tool that has guardrails. 

result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

The email is not professional or has placeholders and will not be sent


## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state." Basically it's an one-time environment that you can do your own experiments without affecting the original coded. 

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [51]:
# Sandbox agents require openai-agents >= 0.14.0 (added in May 2026).
# Force-reinstall to avoid mixed old/new package files, then restart the kernel.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "openai-agents>=0.14.0"])

Looking in links: /var/folders/pn/1wj995516dg3q9_p7h3htplr0000gn/T/tmptsyc_r1w
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.6 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.meta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
msal 1.34.0 requires cryptography<49,>=2.5, but you have cryptography 49.0.0 which is incompatible.
semantic-kernel 1.36.0 requires pydantic!=2.10.0,!=2.10.1,!=2.10.2,!=2.10.3,<2.12,>=2.0, but you have pydantic 2.13.4 which is incompatible.
semantic-kernel 1.36.0 requires websockets<16,>=13, but you have websockets 16.0 which is incompatible.
jsonschema-path 0.3.4 requires referencing<0.37.0, but you have referencing 0.37.0 which is incompatible.
polygon-api-client 1.15.4 requires certifi<2026.0.0,>=2022.5.18, but you have certifi 2026.6.17 which is incompatible.
polygon-api-client 1.15.4 requires websockets<15.0,>=10.3, but you have websockets 16.0 which is incompatible.
pyopenssl 25.3.0 requires cryptography<47,>=45.0.7, but you have cryptography 49.0.0 which is incompatible.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


0

In [52]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [53]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [54]:
CODE_DIR

PosixPath('/Users/duongnguyen/Documents/GitHub/agents/2_openai/code')

In [55]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [56]:
# Create manifest object 

manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [57]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [58]:
agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)

In [59]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

LocalDirReadError: failed to read local dir artifact: /Users/duongnguyen/Documents/GitHub/agents/2_openai/code

## OPTIONAL EXTRA: MCP Teaser!

In [61]:
# MCP is very easy to use within the OpenAI Agent SDK. It allows you to use tools written by other people. You can add those tools to your own agent. 
# Instead of writting the tools with all the Python codes to describe the function and calls for it, or you can just use MCP. You describe the tool and then just use it.  

from agents.mcp import MCPServerStreamableHttp


In [62]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [63]:
agent = Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini")
result = await Runner.run(agent, task)
print(result.final_output)

As of May 2026, the Manifest object in the OpenAI Agents SDK's SandboxAgents feature serves as a mechanism for defining the capabilities, behaviors, and constraints of an agent. It outlines the agent's functionalities, including the actions it can perform, the inputs it requires, and any relevant parameters. This structured representation helps ensure that agents operate within defined boundaries while interacting with their environment, enabling developers to create more predictable and manageable experiences.


In [65]:
# Context7 is an MCP server tool that gives you the latest documentation on AI models. 

params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model="gpt-4o-mini")
    result = await Runner.run(agent, task)

print(result.final_output)

In the OpenAI Agents SDK's SandboxAgents feature, the `Manifest` object plays a crucial role in defining the workspace for a sandbox agent. It specifies the environment in which the agent operates, including user definitions and file access permissions.

Here are the key roles of the `Manifest` object:

1. **User Definition**: It allows you to declare users who will interact with the sandbox. This is essential for configuring permissions effectively.

2. **Entries Configuration**: The `Manifest` contains entries that specify which directories or files are accessible by the declared users. Each entry can assign permissions, such as read or write access, thereby controlling the agent's ability to interact with files.

3. **Permissions Management**: It enables fine-grained control over what actions users can perform on various files or directories, thereby enhancing security and functionality.

4. **Workspace Initialization**: The `Manifest` establishes the initial state and rules for the

And see the traces:

https://platform.openai.com/traces
